In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['pdf.fonttype'] = 42

/home/yuanyuan.fu/bin/miniconda3/envs/iterative_scANVI_May9/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/yuanyuan.fu/bin/miniconda3/envs/iterative_scANVI_May9/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/yuanyuan.fu/bin/miniconda3/envs/iterative_scANVI_May9/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/yuanyuan.fu/bin/miniconda3/envs/iterative_scANVI_May9/lib/python3.11/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/

In [2]:
h21 = sc.read_h5ad("/allen/programs/celltypes/workgroups/rnaseqanalysis/EvoGen/Team/Yuanyuan/astro_h5ad/HMBA_human_Astro_obs_Notes.h5ad")
h21

AnnData object with n_obs × n_vars = 136684 × 36601
    obs: 'Neighborhood', 'Class', 'Subclass', 'Group', 'Cluster', 'cluster_id', 'cell_type_ontology_term', 'load_id', 'donor_id', 'assay', 'assay_ontology_term_id', 'organism', 'organism_ontology_term_id', 'development_stage', 'anatomical_region', 'anatomical_region_merged', 'anatomical_region_ontology_term_id', 'brain_region_ontology_term_id', 'self_reported_sex', 'self_reported_sex_ontology_term_id', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'disease', 'disease_ontology_term_id', 'suspension_type', 'is_primary_data', 'atac_confidently_mapped_read_pairs', 'atac_fraction_of_genome_in_peaks', 'atac_fraction_of_high_quality_fragments_in_cells', 'atac_fraction_of_high_quality_fragments_overlapping_tss', 'atac_fraction_of_high_quality_fragments_overlapping_peaks', 'atac_fraction_of_transposition_events_in_peaks_in_cells', 'atac_mean_raw_read_pairs_per_cell', 'atac_median_high_quality_fragments_per_cell', 'atac

In [3]:
drop_clusters = ["Human-232", "Human-143"] #mixed

mask = ~h21.obs["Cluster"].isin(drop_clusters)
h19 = h21[mask].copy()

print("Remaining cells:", h19.n_obs)
print(h19.obs["cluster"].value_counts())

Remaining cells: 134124
cluster
h-14     58596
h-160    17037
h-485    12908
h-472    12864
h-159    12053
h-31      3350
h-149     2649
h-452     2522
h-231     1998
h-7       1893
h-145     1783
h-450     1643
h-28      1492
h-150     1359
h-152     1322
h-479      243
h-146      172
h-227      162
h-230       78
Name: count, dtype: int64


In [4]:
h19.obs["subGroup"] = pd.Categorical(h19.obs["subGroup"], categories=["GM STR", "GM exSTR", "WM"], ordered=True)

In [5]:
#h19 = sc.read_h5ad("/data/BG_ASC_revision_dataset_input2/HMBA_human_Astro_3subGroups.h5ad")
h19

AnnData object with n_obs × n_vars = 134124 × 36601
    obs: 'Neighborhood', 'Class', 'Subclass', 'Group', 'Cluster', 'cluster_id', 'cell_type_ontology_term', 'load_id', 'donor_id', 'assay', 'assay_ontology_term_id', 'organism', 'organism_ontology_term_id', 'development_stage', 'anatomical_region', 'anatomical_region_merged', 'anatomical_region_ontology_term_id', 'brain_region_ontology_term_id', 'self_reported_sex', 'self_reported_sex_ontology_term_id', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'disease', 'disease_ontology_term_id', 'suspension_type', 'is_primary_data', 'atac_confidently_mapped_read_pairs', 'atac_fraction_of_genome_in_peaks', 'atac_fraction_of_high_quality_fragments_in_cells', 'atac_fraction_of_high_quality_fragments_overlapping_tss', 'atac_fraction_of_high_quality_fragments_overlapping_peaks', 'atac_fraction_of_transposition_events_in_peaks_in_cells', 'atac_mean_raw_read_pairs_per_cell', 'atac_median_high_quality_fragments_per_cell', 'atac

In [6]:
X = h19.raw.X

rows = np.random.choice(X.shape[0], 1000, replace=False)
cols = np.random.choice(X.shape[1], 1000, replace=False)

sub = X[rows][:, cols].toarray()
print(np.allclose(sub, np.round(sub)))
print(sub.max())

True
313.0


### 1.1  downsample

In [7]:
pd.crosstab(h19.obs['subGroup'], h19.obs['donor_id'])


donor_id,H18.30.001,H19.30.004,H20.30.001,H20.30.002,H21.30.004,H23.30.001,H24.30.001,H24.30.003,H24.30.004,H24.30.007
subGroup,,,,,,,,,,
GM STR,2003,2,5380,3708,8040,5813,11356,10737,9353,7447
WM,2002,443,1774,1935,3105,4924,2431,1648,8989,2494


In [9]:
set(h19.obs['subGroup'])

{'GM STR', 'WM', nan}

In [10]:
h19.obs["subGroup"] = h19.obs["subGroup"].fillna("GM exSTR")

In [11]:
pd.crosstab(h19.obs['subGroup'], h19.obs['donor_id'])


donor_id,H18.30.001,H19.30.004,H20.30.001,H20.30.002,H21.30.004,H23.30.001,H24.30.001,H24.30.003,H24.30.004,H24.30.007
subGroup,,,,,,,,,,
GM STR,2003,2,5380,3708,8040,5813,11356,10737,9353,7447
GM exSTR,2905,692,5560,2541,4993,3024,5062,7302,4121,4340
WM,2002,443,1774,1935,3105,4924,2431,1648,8989,2494


In [12]:
def downsample_adata(
    adata,
    group_key,
    donor_key,
    max_cells=1000,
    min_cells=10,
    random_state=123,
    verbose=True
):
    
    rng = np.random.default_rng(random_state)
    keep_indices = []
    
    dropped = 0
    kept = 0

    for donor in sorted(adata.obs[donor_key].unique()):
        adata_donor = adata[adata.obs[donor_key] == donor]
        
        for grp in sorted(adata_donor.obs[group_key].unique()):
            idx = adata_donor.obs[group_key] == grp
            cells = adata_donor.obs[idx].index.values
            
            # filter very low-coverage
            if len(cells) < min_cells:
                dropped += 1
                continue
            
            # downsample
            if len(cells) > max_cells:
                sampled = rng.choice(cells, max_cells, replace=False)
            else:
                sampled = cells
            
            keep_indices.extend(sampled)
            kept += 1

    if verbose:
        print(f"[Downsample] kept groups: {kept}, dropped groups (<{min_cells} cells): {dropped}")
        print(f"[Downsample] total cells kept: {len(keep_indices)}")

    return adata[keep_indices].copy()


In [13]:
h19_subgroup_ds = downsample_adata(
    h19,
    group_key="subGroup",
    donor_key="donor_id",
    max_cells=1000,
    min_cells=10
)


[Downsample] kept groups: 29, dropped groups (<10 cells): 1
[Downsample] total cells kept: 28135


In [14]:
pd.crosstab(h19_subgroup_ds.obs['subGroup'], h19_subgroup_ds.obs['donor_id'])


donor_id,H18.30.001,H19.30.004,H20.30.001,H20.30.002,H21.30.004,H23.30.001,H24.30.001,H24.30.003,H24.30.004,H24.30.007
subGroup,,,,,,,,,,
GM STR,1000,0,1000,1000,1000,1000,1000,1000,1000,1000
GM exSTR,1000,692,1000,1000,1000,1000,1000,1000,1000,1000
WM,1000,443,1000,1000,1000,1000,1000,1000,1000,1000


### 1.2  DEGs across subgroups via pseudobulk DEseq2

In [15]:
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

In [16]:
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

def _get_counts_matrix(adata, layer_key=None):
    if layer_key is not None:
        X = adata.layers[layer_key]
        genes = adata.var_names
    elif adata.raw is not None:
        X = adata.raw.X
        genes = adata.raw.var_names
    else:
        X = adata.X
        genes = adata.var_names

    if issparse(X):
        X = X.toarray()
    X = np.asarray(X)

    if (X < 0).any():
        raise ValueError("Counts contain negative values. DESeq2 expects non-negative counts.")

    if not np.issubdtype(X.dtype, np.integer):
        Xr = np.rint(X)
        if not np.allclose(X, Xr, atol=1e-6):
            raise ValueError("Counts are non-integer-like. Provide raw counts.")
        X = Xr.astype(np.int64)
    else:
        X = X.astype(np.int64)

    return X, genes


def run_subgroup_one_vs_rest_pseudobulk_deseq2(
    adata,
    subgroup_key="subGroup",
    donor_key="donor_id",
    layer_key=None,
    n_cells=3000,
    min_cells_side=10,
    min_donors=2,
    min_gene_counts=5,
    padj_thr=0.05,
    lfc_thr=0.5,
    random_state=123,
    n_cpus=4,
    out_prefix="/home/yuanyuan.fu/hMBA_hu_all/scripts_for_study_paper/real_figures/Subgroup_DEGs_pseudobulk-DESeq2-afterCB",
):
    rng = np.random.default_rng(random_state)
    inference = DefaultInference(n_cpus=n_cpus)

    X, genes = _get_counts_matrix(adata, layer_key=layer_key)
    counts_df = pd.DataFrame(X, index=adata.obs_names, columns=genes)

    obs = adata.obs.copy()
    obs[donor_key] = obs[donor_key].astype(str)
    obs[subgroup_key] = obs[subgroup_key].astype(str)

    donors = sorted(obs[donor_key].unique())
    groups = sorted(obs[subgroup_key].unique())

    all_res = []
    summary_rows = []

    for grp in groups:
        print(f"\n[DESeq2] Running subgroup: {grp} vs rest")

        pb_rows = []
        meta_rows = []
        used_donors = []

        for d in donors:
            cells_target = obs.index[
                (obs[donor_key] == d) &
                (obs[subgroup_key] == grp)
            ]

            cells_rest = obs.index[
                (obs[donor_key] == d) &
                (obs[subgroup_key] != grp)
            ]

            if len(cells_target) < min_cells_side or len(cells_rest) < min_cells_side:
                continue

            if len(cells_target) > n_cells:
                cells_target = rng.choice(cells_target, n_cells, replace=False)

            if len(cells_rest) > n_cells:
                cells_rest = rng.choice(cells_rest, n_cells, replace=False)

            target_sum = counts_df.loc[cells_target].sum(axis=0)
            rest_sum = counts_df.loc[cells_rest].sum(axis=0)

            pb_rows.append(target_sum)
            meta_rows.append({
                donor_key: d,
                "condition": grp,
                "group": grp,
                "comparison": f"{grp}_vs_rest",
                "n_cells": len(cells_target),
                "side": "target",
            })

            pb_rows.append(rest_sum)
            meta_rows.append({
                donor_key: d,
                "condition": "rest",
                "group": grp,
                "comparison": f"{grp}_vs_rest",
                "n_cells": len(cells_rest),
                "side": "rest",
            })

            used_donors.append(d)

        n_donors_used = len(set(used_donors))

        if n_donors_used < min_donors:
            summary_rows.append({
                "group": grp,
                "n_donors_used": n_donors_used,
                "n_sig_genes": 0,
                "status": "skipped_low_donors",
            })
            continue

        counts_pb = pd.DataFrame(pb_rows)
        meta_pb = pd.DataFrame(meta_rows)
        counts_pb.index = [f"s{i}" for i in range(len(counts_pb))]
        meta_pb.index = counts_pb.index

        gene_mask = counts_pb.sum(axis=0) >= min_gene_counts
        counts_pb = counts_pb.loc[:, gene_mask].astype(int)

        dds = DeseqDataSet(
            counts=counts_pb,
            metadata=meta_pb,
            design=f"~ {donor_key} + condition",
            refit_cooks=True,
            inference=inference,
        )
        dds.deseq2()

        stat_res = DeseqStats(
            dds,
            contrast=["condition", grp, "rest"],
            inference=inference,
        )
        stat_res.summary()

        res = stat_res.results_df.copy()
        res["group"] = grp
        res["gene"] = res.index
        res["n_donors_used"] = n_donors_used

        res_filt = res[
            (res["padj"].notna()) &
            (res["padj"] < padj_thr) &
            (res["log2FoldChange"] > lfc_thr)
        ].reset_index(drop=True)

        all_res.append(res_filt)

        summary_rows.append({
            "group": grp,
            "n_donors_used": n_donors_used,
            "n_sig_genes": res_filt.shape[0],
            "status": "ok",
        })

    df_all = pd.concat(all_res, ignore_index=True) if all_res else pd.DataFrame()
    df_summary = pd.DataFrame(summary_rows)

    df_all.to_csv(f"{out_prefix}.csv", index=False)
    df_summary.to_csv(f"{out_prefix}_summary.csv", index=False)

    return df_all, df_summary

In [17]:
df_subgroup_cb, df_subgroup_cb_summary = run_subgroup_one_vs_rest_pseudobulk_deseq2(
    adata=h19_subgroup_ds,
    subgroup_key="subGroup",
    donor_key="donor_id",
    layer_key=None,
    n_cells=3000,
    min_cells_side=10,
    min_donors=2,
    min_gene_counts=5,
    padj_thr=0.05,
    lfc_thr=0.5,
    random_state=123,
    n_cpus=4,
    out_prefix="/home/yuanyuan.fu/hMBA_hu_all/scripts_for_study_paper/real_figures/Subgroup_DEGs_pseudobulk-DESeq2",
)


[DESeq2] Running subgroup: GM STR vs rest


Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 34.20 seconds.

Fitting dispersion trend curve...
... done in 0.75 seconds.

Fitting MAP dispersions...
... done in 45.98 seconds.

Fitting LFCs...
... done in 34.79 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 8.77 seconds.



Log2 fold change & Wald test p-value: condition GM STR vs rest
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1   14.562436        0.320757  0.266668  1.202835  0.229040  0.365335
AL627309.3    0.468478        0.965767  1.824591  0.529306  0.596593       NaN
AL627309.5   40.273141       -0.105110  0.169960 -0.618442  0.536284  0.671336
AP006222.2    0.531919        1.182171  1.518070  0.778733  0.436137       NaN
AC114498.1    0.825368        1.758261  1.545490  1.137672  0.255257       NaN
...                ...             ...       ...       ...       ...       ...
AC004556.3    2.480848        0.272952  0.793596  0.343943  0.730889  0.822658
AC171558.1    0.443118       -1.062063  1.681678 -0.631550  0.527681       NaN
AC007325.1    0.656608       -0.547043  1.285935 -0.425405  0.670542       NaN
AC007325.4   24.523016       -0.577520  0.211955 -2.724728  0.006435  0.019652
AC007325.2  147.638406        0.117511  0.136512  0.860808  0.389344

...

Log2 fold change & Wald test p-value: condition GM exSTR vs rest
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1   12.537705       -0.350196  0.259924 -1.347303  0.177883  0.400561
AL627309.3    0.458242        0.107029  1.563607  0.068450  0.945427       NaN
AL627309.5   35.467470        0.093926  0.175671  0.534671  0.592877  0.792980
AP006222.2    0.452632       -0.362710  1.864022 -0.194585  0.845718       NaN
AC114498.1    0.554895       -0.099868  1.264391 -0.078985  0.937044       NaN
...                ...             ...       ...       ...       ...       ...
AC171558.1    0.479550        0.847220  1.243688  0.681216  0.495735       NaN
AC023491.2    0.274696        1.254499  2.684208  0.467363  0.640240       NaN
AC007325.1    0.733475        0.042828  1.164022  0.036793  0.970650       NaN
AC007325.4   25.691473        0.345785  0.181785  1.902168  0.057149  0.186979
AC007325.2  133.013916        0.223658  0.107712  2.076443  0.0378

Fitting size factors...
... done in 0.03 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 33.84 seconds.

Fitting dispersion trend curve...
... done in 0.72 seconds.

Fitting MAP dispersions...
... done in 46.34 seconds.

Fitting LFCs...
... done in 35.48 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 8.80 seconds.



Log2 fold change & Wald test p-value: condition WM vs rest
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1   12.890270        0.044882  0.249171  0.180125  0.857055  0.915070
AL627309.3    0.459803        0.097636  1.957666  0.049874  0.960223       NaN
AL627309.5   35.840707        0.026147  0.156322  0.167265  0.867162  0.921894
AP006222.2    0.471794        0.629477  1.500267  0.419577  0.674795       NaN
AC114498.1    0.546496       -0.857205  1.457199 -0.588255  0.556361       NaN
...                ...             ...       ...       ...       ...       ...
AC171558.1    0.465397        0.919804  1.398547  0.657685  0.510741       NaN
AC023491.2    0.252210        0.800457  2.756263  0.290414  0.771500       NaN
AC007325.1    0.917693        1.153603  1.016037  1.135394  0.256210       NaN
AC007325.4   25.258298        0.161782  0.179999  0.898794  0.368762  0.535193
AC007325.2  124.078556       -0.339598  0.103293 -3.287699  0.001010  0.